# Trend Event Detector（V1 研究标签）

从 1 分钟 K 线里找出「未来 H 分钟开窗即顺着走、幅度够大、路径够顺」的片段，去重成独立 Trend Start，并画图给人眼检查。

**这是用了未来信息的离线标签，不是交易信号。** 逻辑在 `src/qtrader/labels/trend_events.py` 和 `src/qtrader/experiments/trend_collect.py`。本 notebook 只改参数、跑、看图。

灰色 = 趋势开始前；彩色 = 标签用的未来 H 分钟；黄色 = 再观察 `post_extra` 分钟。


In [1]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "src" / "qtrader").is_dir() and (_p / "config").is_dir():
        REPO_ROOT = _p
        break
else:
    raise ModuleNotFoundError("Cannot find the qtrader repo")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from qtrader.experiments.trend_collect import TrendCollectConfig, collect_trend_events
from qtrader.labels.trend_events import TrendDetectConfig
from qtrader.viz.trend_events import trend_event_chart

print("cwd =", Path.cwd())


cwd = /Users/zihao/work/quant_dev


## 1. 参数

改这一格再重跑收集。阈值是 V1 heuristic，不是搜索结果。开窗后一旦跌破起点，此前顺着走的幅度必须更大（不是只看第一根收盘多 1 个 tick）。
`trend_start` 是该分钟 K 线的**结束时间**（仓库里 K 线时间戳是开盘时间，事件表里 `bar_open` 可用来对上蜡烛）。


In [6]:
CFG = TrendCollectConfig(
    symbols=("QQQ",),
    start="2025-08-01",
    end="2026-09-09",
    feed="iex",
    timeframe="1Min",
    detect=TrendDetectConfig(
        horizon=30,            # H，必须 > 20 的默认落在这里
        vol_lookback=120,      # sigma 只用 t 及以前，且不跨 session / 缺口
        sigma_floor=1e-6,
        z_threshold=1.5,
        er_threshold=0.40,
        mae_threshold=0.50,
        require_first_bar_aligned=True,  # 跌破起点前，顺向幅度必须更大
        cooldown_bars=None,    # None = 等于 H
        bar_minutes=1,
    ),
    output_dir="results/trend_events",
    pre_window=40,
    post_extra=10,
    gallery_n=10,
    overview_per_side=6,
    random_seed=0,
    borderline_z_max=1.8,
    download_if_missing=True,
)
CFG.detect


TrendDetectConfig(horizon=30, vol_lookback=120, sigma_floor=1e-06, z_threshold=1.5, er_threshold=0.4, mae_threshold=0.5, require_first_bar_aligned=True, cooldown_bars=None, bar_minutes=1)

## 2. 收集：检测 → 去重 → 写表和图

本地缺 K 线会下载。每个 symbol 独立计算，不 forward fill 缺失分钟。


In [7]:
result = collect_trend_events(CFG)
print(result.output_dir)
print((result.output_dir / "summary.txt").read_text())


results/trend_events
Trend event detector — V1 heuristic labels (use future data)
bars: 106,605   valid (sigma + future window): 33,669
candidates: UP 396  DOWN 376
events:     UP 90  DOWN 97  rate 0.5554% of valid bars
events/day: mean=1.56  median=1.00

ZTrend: mean=-0.044  median=-1.518  std=2.133  q10=-2.314  q25=-1.853  q50=-1.518  q75=1.814  q90=2.328
ER: mean=0.490  median=0.469  std=0.080  q10=0.411  q25=0.429  q50=0.469  q75=0.527  q90=0.601
MAE_norm: mean=0.024  median=0.000  std=0.059  q10=0.000  q25=0.000  q50=0.000  q75=0.000  q90=0.116
MFE_norm: mean=2.097  median=1.914  std=0.630  q10=1.575  q25=1.686  q50=1.914  q75=2.296  q90=2.777
candidate_run_length: mean=1.647  median=1.000  std=1.123  q10=1.000  q25=1.000  q50=1.000  q75=2.000  q90=3.000
by symbol: {"QQQ": {"events": 187, "up": 90, "down": 97}}

These labels look into the future H-minute window. They are not a live trading signal.



## 3. 事件表


In [8]:
print("candidates", len(result.candidates), "events", len(result.events))
cols = [
    "symbol", "trend_start", "direction", "ztrend", "ER", "MAE_norm",
    "MFE_norm", "future_return", "candidate_run_length", "overlaps_opposite",
]
result.events[cols].head(20)


candidates 772 events 187


,symbol,trend_start,direction,ztrend,ER,MAE_norm,MFE_norm,future_return,candidate_run_length,overlaps_opposite
0,QQQ,2025-08-20 15:54:00+00:00,1,1.519103,0.516581,0.121144,1.544936,0.004687,1,False
1,QQQ,2025-08-20 17:47:00+00:00,-1,-1.804200,0.429392,0.000000,2.034331,-0.003898,1,False
2,QQQ,2025-09-03 19:18:00+00:00,1,1.606238,0.688742,0.000000,1.606238,0.002923,2,False
3,QQQ,2025-09-17 18:54:00+00:00,1,2.950453,0.501922,0.000000,2.950453,0.010083,5,False
4,QQQ,2025-09-18 17:09:00+00:00,-1,-1.575026,0.500968,0.000000,1.826608,-0.002152,1,False
5,QQQ,2025-09-25 17:28:00+00:00,-1,-1.609984,0.461833,0.000000,1.609984,-0.003118,2,False
6,QQQ,2025-09-25 18:05:00+00:00,1,2.646611,0.557975,0.000000,2.734600,0.005077,2,False
7,QQQ,2025-10-27 19:27:00+00:00,1,3.541185,0.426095,0.000000,3.950468,0.001928,1,False
8,QQQ,2025-10-30 15:48:00+00:00,1,1.523674,0.599039,0.000000,1.572958,0.004652,1,False
9,QQQ,2025-10-31 15:46:00+00:00,-1,-2.538773,0.435818,0.000000,2.572221,-0.005429,1,False


## 4. 抽样本：最强 / 随机 / 边界

HTML 在 `results/trend_events/{strongest,random,borderline}/`，总览 `trend_overview.html`。
下面内嵌最强的几张，改 `gallery_n` 后重跑收集即可。


In [5]:
sample = result.strongest(50)
for _, event in sample.iterrows():
    fig = trend_event_chart(
        result.bars[event["symbol"]], event,
        pre_window=CFG.pre_window,
        post_extra=CFG.post_extra,
    )
    fig.show()
